# 🛡️ Commit Guardian — Google Colab GPU + 5 ТБ Google Drive Кэш

Запускает **Ollama (`qwen2.5-coder:32b` / `14b`)** и весь **Commit Guardian (Web UI + API + Dashboard + PDF + Telegram Bot)** на GPU в Google Colab, не нагружая твой Mac.

### Что сделать за 3 клика:
1. В меню сверху выбери **Среда выполнения → Сменить среду выполнения → GPU** (**A100** или **L4** для `qwen2.5-coder:32b`, либо **T4** для `qwen2.5-coder:14b`).
2. Нажми **Среда выполнения → Выполнить все** (`Runtime → Run all`). При первом запуске нажми **Разрешить** в окне Google Drive — веса модели (`9–20 ГБ`) и PDF-отчёты сохранятся на твоих **5 ТБ Google Drive**.
3. В **Ячейке 4** появятся:
   - Готовая HTTPS-ссылка **`https://...trycloudflare.com`** на веб-консоль Commit Guardian (и `/dashboard`) для живого демо жюри;
   - Если поставить галочку **`EXPOSE_OLLAMA_TUNNEL`** — отдельная HTTPS-ссылка на GPU Ollama, чтобы подключить локальный Mac или Railway к GPU в Colab (`OLLAMA_BASE_URL`).

> **Как устроен быстрый кэш Drive ↔ SSD:** Google Drive работает по FUSE, из-за чего прямое чтение весов Ollama (`mmap`) тормозит или падает. Поэтому мастер-копия весов хранится на 5 ТБ Drive (`/content/drive/MyDrive/CommitGuardian/ollama_models`), при старте сессии быстро переносится на локальный NVMe SSD (`/content/ollama_models`), а после первого скачивания новые слои автоматически дописываются обратно на Drive.

> **Секреты (опционально):** токены `TELEGRAM_BOT_TOKEN` и `GITHUB_TOKEN` можно положить в левую панель Colab **🔑 Секреты** (или вставить в поля Ячейки 4 перед запуском — они сохраняются только в `.env` внутри временной VM Colab с правами `0600`).

In [ ]:
#@title 1. Подключение 5 ТБ Google Drive, выбор GPU-модели и перенос кэша на SSD
import os
import shutil
import subprocess
import time
from pathlib import Path

MODEL_CHOICE = "auto"  #@param ["auto", "qwen2.5-coder:32b", "qwen2.5-coder:14b", "qwen2.5-coder:7b", "qwen2.5-coder:3b"]
DRIVE_BASE_DIR = "/content/drive/MyDrive/CommitGuardian"  #@param {type:"string"}

if Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(f"Не удалось подключить Google Drive: {exc}") from exc
else:
    print("Запуск вне Google Colab — используем локальный путь:", DRIVE_BASE_DIR)

drive_base = Path(DRIVE_BASE_DIR)
drive_models = drive_base / "ollama_models"
drive_reports = drive_base / "reports"
drive_data = drive_base / "data"
local_models = Path("/content/ollama_models")
log_dir = Path("/content/cg_logs")

for directory in (drive_models / "blobs", drive_models / "manifests", drive_reports, drive_data, local_models, log_dir):
    directory.mkdir(parents=True, exist_ok=True)


def detect_gpu() -> tuple[str, int]:
    """Return GPU name and total VRAM in MiB."""
    try:
        proc = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True,
        )
        first_line = proc.stdout.strip().splitlines()[0]
        name, vram_str = [part.strip() for part in first_line.split(",", 1)]
        return name, int(float(vram_str))
    except Exception:
        return "CPU (GPU не включён)", 0


gpu_name, vram_mb = detect_gpu()

if MODEL_CHOICE == "auto":
    if vram_mb >= 36000:
        selected_model = "qwen2.5-coder:32b"  # A100 40/80GB, H100
    elif vram_mb >= 22000:
        selected_model = "qwen2.5-coder:32b"  # L4 24GB (с q8_0 KV-кэшем и flash attention)
    elif vram_mb >= 14000:
        selected_model = "qwen2.5-coder:14b"  # T4 16GB — штатная модель Commit Guardian
    elif vram_mb >= 6000:
        selected_model = "qwen2.5-coder:7b"
    else:
        selected_model = "qwen2.5-coder:3b"
else:
    selected_model = MODEL_CHOICE

os.environ["CG_MODEL_NAME"] = selected_model
os.environ["CG_DRIVE_BASE"] = str(drive_base)
os.environ["CG_DRIVE_MODELS"] = str(drive_models)
os.environ["CG_DRIVE_REPORTS"] = str(drive_reports)
os.environ["CG_DRIVE_DATA"] = str(drive_data)
os.environ["CG_LOCAL_OLLAMA_MODELS"] = str(local_models)
os.environ["OLLAMA_MODELS"] = str(local_models)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"


def sync_ollama_store(src_root: Path, dst_root: Path, label: str) -> None:
    """Copy missing/updated Ollama blobs and manifests between Drive and local NVMe SSD."""
    if not src_root.exists():
        return
    copied_files = 0
    copied_bytes = 0
    started = time.time()

    src_blobs = src_root / "blobs"
    dst_blobs = dst_root / "blobs"
    dst_blobs.mkdir(parents=True, exist_ok=True)
    if src_blobs.exists():
        for blob in src_blobs.iterdir():
            if not blob.is_file() or blob.name.endswith("-partial"):
                continue
            target = dst_blobs / blob.name
            src_size = blob.stat().st_size
            if target.exists() and target.stat().st_size == src_size:
                continue
            size_gb = src_size / (1024 ** 3)
            if size_gb >= 0.05:
                print(f"[{label}] Копирую {blob.name[:24]}... ({size_gb:.2f} ГБ)")
            shutil.copy2(blob, target)
            copied_files += 1
            copied_bytes += src_size

    src_manifests = src_root / "manifests"
    dst_manifests = dst_root / "manifests"
    if src_manifests.exists():
        shutil.copytree(src_manifests, dst_manifests, dirs_exist_ok=True)

    elapsed = time.time() - started
    if copied_files:
        print(f"[{label}] Перенесено файлов: {copied_files} ({copied_bytes / (1024 ** 3):.2f} ГБ) за {elapsed:.1f} с")
    else:
        print(f"[{label}] Новых бинарных слоёв для копирования нет (время проверки: {elapsed:.1f} с)")


sync_ollama_store(drive_models, local_models, "Drive → SSD")

drive_usage = shutil.disk_usage(drive_base)
ssd_usage = shutil.disk_usage("/content")

print("-" * 64)
print(f"GPU:            {gpu_name} ({vram_mb / 1024:.1f} ГБ VRAM)" if vram_mb else f"GPU:            {gpu_name}")
if vram_mb == 0:
    print("⚠️  ВНИМАНИЕ: GPU не активен! Открой «Среда выполнения → Сменить среду выполнения → GPU (L4/A100/T4)».")
print(f"Модель:         {selected_model}")
print(f"5 ТБ Drive:     свободно {drive_usage.free / (1024 ** 3):.1f} ГБ из {drive_usage.total / (1024 ** 3):.1f} ГБ")
print(f"Colab SSD:      свободно {ssd_usage.free / (1024 ** 3):.1f} ГБ из {ssd_usage.total / (1024 ** 3):.1f} ГБ")
print(f"Кэш отчётов:    {drive_reports}")
print(f"Метрики SQLite: {drive_data / 'commit_guardian.db'}")
print("-" * 64)

In [ ]:
#@title 2. Установка Ollama, скачивание/прогрев модели и сохранение весов на 5 ТБ Drive
import json
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

model_name = os.environ["CG_MODEL_NAME"]
local_models = Path(os.environ["CG_LOCAL_OLLAMA_MODELS"])
drive_models = Path(os.environ["CG_DRIVE_MODELS"])
log_dir = Path("/content/cg_logs")
log_dir.mkdir(parents=True, exist_ok=True)

# Системные пакеты: zstd нужен установщику Ollama, fonts-dejavu-core — для кириллицы в PDF-отчётах
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd", "pciutils", "fonts-dejavu-core"],
    check=True,
)

if shutil.which("ollama") is None:
    print("Устанавливаю Ollama...")
    installer_path = log_dir / "ollama_install.sh"
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer_path)
    subprocess.run(["sh", str(installer_path)], check=True)
else:
    print("Ollama уже установлена:", shutil.which("ollama"))

subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)

ollama_env = os.environ.copy()
ollama_env.update(
    {
        "OLLAMA_MODELS": str(local_models),
        "OLLAMA_HOST": "127.0.0.1:11434",
        "OLLAMA_FLASH_ATTENTION": "1",
        "OLLAMA_KV_CACHE_TYPE": "q8_0",
        "OLLAMA_NUM_PARALLEL": "1",
        "OLLAMA_KEEP_ALIVE": "-1",
    }
)

with open(log_dir / "ollama.log", "ab") as ollama_log:
    subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_env,
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama не поднялась за 30 секунд — проверь /content/cg_logs/ollama.log")

print(f"Ollama запущена. Проверяю/скачиваю модель {model_name}...")
subprocess.run(["ollama", "pull", model_name], env=ollama_env, check=True)

# Сохраняем новые скачанные веса обратно на 5 ТБ Drive, чтобы при следующем запуске не качать заново
if "sync_ollama_store" in globals():
    sync_ollama_store(local_models, drive_models, "SSD → Drive")

total_cached_gb = sum(p.stat().st_size for p in drive_models.rglob("*") if p.is_file()) / (1024 ** 3)
print(f"Веса сохранены на Google Drive ({drive_models}): {total_cached_gb:.2f} ГБ")

print(f"Прогреваю {model_name} в GPU VRAM...")
payload = json.dumps(
    {
        "model": model_name,
        "prompt": "Reply with OK only.",
        "stream": False,
        "keep_alive": -1,
        "options": {"num_predict": 8},
    }
).encode("utf-8")
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)
t0 = time.time()
with urllib.request.urlopen(req, timeout=180) as resp:
    data = json.loads(resp.read().decode("utf-8"))
eval_count = data.get("eval_count", 0)
eval_ns = max(data.get("eval_duration", 1), 1)
tps = eval_count / (eval_ns / 1e9)
print(f"Готово за {time.time() - t0:.1f} с · ответ: {data.get('response', '').strip()!r} · скорость: {tps:.1f} tok/s")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"], check=False)

In [ ]:
#@title 3. Клонирование Commit Guardian с GitHub, зависимости и сборка Next.js консоли
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/madiyarmoldakhmet-ai/cybersecyritybot.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

repo_dir = Path("/content/cybersecyritybot")

if (repo_dir / ".git").exists():
    print(f"Обновляю {repo_dir} ({BRANCH})...")
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print(f"Клонирую {REPO_URL} ({BRANCH})...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)

print("Устанавливаю Python-зависимости...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"], check=True)
req_file = repo_dir / "requirements.txt"
pip_res = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(req_file)])
if pip_res.returncode != 0:
    # На случай среды с Python <3.12 убираем опциональный пакет strix-agent (не импортируется в рантайме)
    print("Повторяю установку зависимостей без опционального пакета strix-agent...")
    filtered_lines = [
        line for line in req_file.read_text(encoding="utf-8").splitlines()
        if not line.strip().startswith("strix-agent")
    ]
    fallback_req = Path("/content/cg_logs/requirements_colab.txt")
    fallback_req.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(fallback_req)], check=True)


def ensure_node20_plus() -> str:
    """Ensure Node.js >= 20 is available for Next.js static export."""
    node_bin = shutil.which("node")
    if node_bin:
        ver_out = subprocess.run([node_bin, "-v"], capture_output=True, text=True).stdout.strip()
        match = re.match(r"v(\d+)\.", ver_out)
        if match and int(match.group(1)) >= 20:
            return ver_out

    print("Ставлю Node.js v22 LTS для сборки веб-консоли...")
    with urllib.request.urlopen("https://nodejs.org/dist/index.json", timeout=30) as resp:
        releases = json.loads(resp.read().decode("utf-8"))
    v22_release = next(r["version"] for r in releases if r["version"].startswith("v22.") and r.get("lts"))
    archive_name = f"node-{v22_release}-linux-x64.tar.xz"
    archive_path = Path("/content/cg_logs") / archive_name
    urllib.request.urlretrieve(f"https://nodejs.org/dist/{v22_release}/{archive_name}", archive_path)

    node_root = Path("/usr/local/node22")
    shutil.rmtree(node_root, ignore_errors=True)
    node_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:xz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(node_root, filter="data")
        else:
            tar.extractall(node_root)

    extracted_dir = node_root / f"node-{v22_release}-linux-x64"
    for binary in ("node", "npm", "npx"):
        dst = Path("/usr/local/bin") / binary
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        dst.symlink_to(extracted_dir / "bin" / binary)

    return subprocess.run(["node", "-v"], capture_output=True, text=True, check=True).stdout.strip()


node_version = ensure_node20_plus()
print("Node.js:", node_version)

commit_sha = subprocess.run(
    ["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
web_dir = repo_dir / "aegis_web"
built_marker = web_dir / ".built_sha"

if not (web_dir / "out" / "index.html").exists() or not built_marker.exists() or built_marker.read_text().strip() != commit_sha:
    print(f"Собираю Next.js консоль (commit {commit_sha})...")
    subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=web_dir, check=True)
    subprocess.run(["npm", "run", "build"], cwd=web_dir, check=True)
    built_marker.write_text(commit_sha, encoding="utf-8")
else:
    print(f"Next.js консоль уже собрана для commit {commit_sha}")

os.environ["CG_REPO_DIR"] = str(repo_dir)
os.environ["CG_COMMIT_SHA"] = commit_sha
check_proc = subprocess.run(
    [sys.executable, "-c", "from aegis_server.main import app; print('Backend OK:', app.title)"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
)
if check_proc.returncode != 0:
    print(check_proc.stderr)
    raise RuntimeError("Ошибка импорта backend — смотри вывод выше")
print(check_proc.stdout.strip())

In [ ]:
#@title 4. Запуск сайта Commit Guardian + публичная HTTPS-ссылка (и опциональный туннель Ollama)
import json
import os
import re
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

ENVIRONMENT = "production"  #@param ["production", "development"]
EXPOSE_OLLAMA_TUNNEL = False  #@param {type:"boolean"}
TELEGRAM_BOT_TOKEN = ""  #@param {type:"string"}
ALLOWED_TELEGRAM_USERS = ""  #@param {type:"string"}
DEFAULT_TELEGRAM_CHAT_ID = ""  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
GITHUB_WEBHOOK_SECRET = ""  #@param {type:"string"}
RUN_IN_FOREGROUND = True  #@param {type:"boolean"}

repo_dir = Path(os.environ.get("CG_REPO_DIR", "/content/cybersecyritybot"))
log_dir = Path("/content/cg_logs")
log_dir.mkdir(parents=True, exist_ok=True)
model_name = os.environ.get("CG_MODEL_NAME", "qwen2.5-coder:14b")
drive_reports = os.environ.get("CG_DRIVE_REPORTS", "/content/drive/MyDrive/CommitGuardian/reports")
drive_data = Path(os.environ.get("CG_DRIVE_DATA", "/content/drive/MyDrive/CommitGuardian/data"))
drive_data.mkdir(parents=True, exist_ok=True)


def resolve_secret(param_value: str, secret_name: str) -> str:
    """Read parameter first, then fall back to Google Colab Secrets sidebar."""
    if param_value and param_value.strip():
        return param_value.strip()
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        return val.strip() if val else ""
    except Exception:
        return ""


tg_token = resolve_secret(TELEGRAM_BOT_TOKEN, "TELEGRAM_BOT_TOKEN")
tg_users = resolve_secret(ALLOWED_TELEGRAM_USERS, "ALLOWED_TELEGRAM_USERS")
tg_chat = resolve_secret(DEFAULT_TELEGRAM_CHAT_ID, "DEFAULT_TELEGRAM_CHAT_ID")
gh_token = resolve_secret(GITHUB_TOKEN, "GITHUB_TOKEN")
gh_webhook_secret = resolve_secret(GITHUB_WEBHOOK_SECRET, "GITHUB_WEBHOOK_SECRET")

if tg_token and ENVIRONMENT == "production" and not tg_users:
    print("⚠️  Подсказка: в режиме ENVIRONMENT=production Telegram-бот требует ALLOWED_TELEGRAM_USERS (твой числовой user_id), иначе он будет отклонять команды.")

cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    print("Скачиваю Cloudflare Tunnel (cloudflared)...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

# Останавливаем старые процессы при перезапуске ячейки
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
subprocess.run(["pkill", "-f", "uvicorn aegis_server.main:app"], check=False)
time.sleep(1)


def start_quick_tunnel(args: list[str], log_path: Path) -> str:
    """Launch cloudflared quick tunnel and parse the assigned HTTPS URL."""
    if log_path.exists():
        log_path.unlink()
    with open(log_path, "ab") as log_file:
        subprocess.Popen(
            [str(cf_bin), "tunnel", *args, "--no-autoupdate"],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    for _ in range(30):
        if log_path.exists():
            text = log_path.read_text(encoding="utf-8", errors="ignore")
            match = pattern.search(text)
            if match:
                return match.group(0)
        time.sleep(1)
    raise RuntimeError(f"Cloudflare tunnel не выдал URL за 30с — смотри {log_path}")


app_public_url = start_quick_tunnel(["--url", "http://127.0.0.1:8000"], log_dir / "cf_app.log")
ollama_public_url = ""
if EXPOSE_OLLAMA_TUNNEL:
    ollama_public_url = start_quick_tunnel(
        ["--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
        log_dir / "cf_ollama.log",
    )

env_lines = [
    f"ENVIRONMENT={ENVIRONMENT}",
    "ALLOW_CLOUD_LLM=false",
    "USE_LOCAL_LLM=true",
    "LLM_PROVIDER=ollama",
    "OLLAMA_BASE_URL=http://127.0.0.1:11434",
    f"OLLAMA_MODEL={model_name}",
    "AEGIS_OLLAMA_BASE_URL=http://127.0.0.1:11434/v1",
    f"AEGIS_MODEL={model_name}",
    f"SCAN_OUTPUT_DIR={drive_reports}",
    f"METRICS_DB_PATH={drive_data / 'commit_guardian.db'}",
    "REPORT_FONT_PATH=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    f"WEB_APP_URL={app_public_url}",
    f"CORS_ALLOW_ORIGINS={app_public_url}",
]
if tg_token:
    env_lines.append(f"TELEGRAM_BOT_TOKEN={tg_token}")
if tg_users:
    env_lines.append(f"ALLOWED_TELEGRAM_USERS={tg_users}")
if tg_chat:
    env_lines.append(f"DEFAULT_TELEGRAM_CHAT_ID={tg_chat}")
if gh_token:
    env_lines.append(f"GITHUB_TOKEN={gh_token}")
if gh_webhook_secret:
    env_lines.append(f"GITHUB_WEBHOOK_SECRET={gh_webhook_secret}")

env_file = repo_dir / ".env"
env_file.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
os.chmod(env_file, 0o600)

uvicorn_log_path = log_dir / "uvicorn.log"
if uvicorn_log_path.exists():
    uvicorn_log_path.unlink()
with open(uvicorn_log_path, "ab") as uvicorn_log:
    uvicorn_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "aegis_server.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=repo_dir,
        stdout=uvicorn_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

for _ in range(45):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            health_payload = json.loads(resp.read().decode("utf-8"))
            if health_payload.get("status") == "healthy":
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Commit Guardian backend не ответил на /health — смотри /content/cg_logs/uvicorn.log")

print("=" * 72)
print("🛡️  COMMIT GUARDIAN ЗАПУЩЕН НА GOOGLE COLAB GPU")
print("=" * 72)
print(f"🌐 Веб-консоль (главная):  {app_public_url}")
print(f"📊 Air-gap Dashboard:      {app_public_url}/dashboard")
print(f"📚 Swagger API docs:       {app_public_url}/docs")
print(f"🤖 Активная GPU-модель:    {health_payload.get('ollama_model')} (ollama_online={health_payload.get('ollama_online')})")
print(f"🔒 Режим приватности:      air_gapped={health_payload.get('air_gapped')} ({ENVIRONMENT})")
print(f"💾 Отчёты и база на Drive: {drive_reports}")
if ollama_public_url:
    print("-" * 72)
    print("🔌 Внешний GPU-туннель Ollama для твоего Mac или Railway (.env):")
    print(f"OLLAMA_BASE_URL={ollama_public_url}")
    print(f"OLLAMA_MODEL={model_name}")
    print(f"AEGIS_OLLAMA_BASE_URL={ollama_public_url}/v1")
    print(f"AEGIS_MODEL={model_name}")
    print("⚠️  Этот URL временный (действует пока работает ячейка). Не публикуй его в открытых репозиториях.")
print("=" * 72)

if RUN_IN_FOREGROUND:
    print("Стриминг логов uvicorn (жми Стоп ⏹️ в Colab, если захочешь остановить сервер)...")
    offset = 0
    try:
        while True:
            if uvicorn_log_path.exists():
                with open(uvicorn_log_path, "r", encoding="utf-8", errors="ignore") as f:
                    f.seek(offset)
                    chunk = f.read()
                    if chunk:
                        print(chunk, end="", flush=True)
                        offset = f.tell()
            if uvicorn_proc.poll() is not None:
                print(f"\n⚠️ Процесс uvicorn завершился с кодом {uvicorn_proc.returncode}")
                break
            time.sleep(2)
    except KeyboardInterrupt:
        print("\n⏹️ Остановка стриминга логов по нажатию Stop.")